## はじめに

このノートブックでは、Snowflake Cortex Agentを作成し、構造化データと非構造化データを横断した自然言語分析を実現します。

**主な処理内容:**
- Cortex Agent（GLACIER_ANALYTICS_AGENT）の作成
- Semantic Viewツールの追加（売上・顧客分析用）
- Cortex Searchツールの追加（FAQ・マニュアル・音声ログ・SNS検索用）
- エージェントの動作確認

**Cortex Agentとは:**
- 構造化データ（Semantic View経由のSQL生成）と非構造化データ（Cortex Search検索）を統合
- LLMベースのオーケストレーションで適切なツールを自動選択
- Snowflake Intelligenceとして公開可能

---

### 本ハンズオンの進め方

| ステップ | 作成方法 | 内容 |
|---------|---------|------|
| Step 1 | Notebooks | Cortex Searchツール3つを含むエージェントを作成 |
| Step 2 | **AI Studio（GUI）** | Semantic Viewツールを追加（GUI体験 1/2） |
| Step 3 | **AI Studio（GUI）** | Cortex Searchツール（SNS）を追加して完成（GUI体験 2/2） |
| Step 4 | AI Studio | 動作確認・分析実践 |

> **💡 ポイント**  
> Notebooksで先にベースを完成させ、最後にAI StudioでGUI操作を体験しながら仕上げます。

---

### ⚠️ 重要: Cortex Agentの制約について

Cortex Agentには **ALTER AGENTコマンドが存在しません**。  
ツールの追加・変更方法は以下の4つです：

1. **CREATE AGENT時に定義**: SPECIFICATION内にすべてのツールを含める
2. **CREATE OR REPLACE AGENT**: ツールを追加・変更するたびに再作成
3. **AI Studio（GUI）**: ビジュアルでツールを追加・編集
4. **REST API**: プログラムからエージェントを更新（[参考ドキュメント](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-rest-api)）

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. 前提条件の確認

Cortex Agentを作成する前に、以下のオブジェクトが存在することを確認します。

**必要なオブジェクト:**
- Semantic View: `EC_ANALYSIS_SEMANTIC_VIEW`
- Cortex Search Services: `SEARCH_FAQ`, `SEARCH_OPERATION_MANUALS`, `SEARCH_VOICE_LOGS`, `SEARCH_SNS_MENTIONS`

In [ ]:
-- ============================================================================
-- Semantic Viewの確認
-- ============================================================================
SHOW SEMANTIC VIEWS IN SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 2. Cortex Agentの作成（Cortex Searchツール3つ付き）

Cortex Agentを作成し、3つのCortex Searchツールを設定します。  
残りのツール（Semantic View、SNS検索）はAI Studioで追加します。

**エージェント設定:**
- オブジェクト名: `GLACIER_ANALYTICS_AGENT`
- 表示名: `GLACIER分析エージェント`
- モデル: `auto`（最新のモデルが自動選択）

**含まれるツール（3つ）:**
- `FAQ_Search`: FAQドキュメント検索
- `Operation_Manual_Search`: 業務マニュアル検索
- `Voice_Log_Search`: 音声ログ検索

**AI Studioで追加するツール（2つ）:**
- `EC_Sales_Customer_Analysis`: Semantic View（売上・顧客分析）
- `SNS_Mention_Search`: SNS投稿検索

In [ ]:
-- ============================================================================
-- Cortex Agent の作成（Cortex Searchツール3つ付き）
-- ============================================================================
CREATE OR REPLACE AGENT GLACIER_ANALYTICS_AGENT
  COMMENT = 'GlacierStyle ECサイトの売上・顧客・VoC分析を自然言語で行うエージェントです。'
  PROFILE = '{"display_name": "GLACIER分析エージェント", "color": "blue"}'
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  orchestration: |
    あなたはGlacierStyle ECサイトの分析アシスタントです。
    ユーザーの質問に対して、以下の手順で適切なツールを選択してください。
    
    1. 質問の種類を判断する
       - 売上・注文・顧客・商品に関する数値分析 → EC_Sales_Customer_Analysis（Semantic View）を使用
       - 返品・配送・支払いなどのFAQ → FAQ_Search を使用
       - 業務手順・対応方法 → Operation_Manual_Search を使用
       - 過去の問い合わせ事例 → Voice_Log_Search を使用
       - SNSの評判・口コミ → SNS_Mention_Search を使用
    
    2. 複合的な質問の場合は、複数のツールを順番に使用する
    
    3. 検索結果が不十分な場合は、別のツールを試すか、ユーザーに追加情報を求める
    
    4. データの期間に注意する
       - 売上・注文データは2024年のデータです
       - 「今月」「先月」と言われた場合は、2024年12月・11月として解釈してください
  
  response: |
    以下のルールに従って応答してください。
    
    【口調・スタイル】
    - 丁寧語（です・ます調）で回答する
    - 専門用語は必要に応じて簡単な説明を添える
    - 回答は簡潔にまとめつつ、必要な情報は漏らさない
    
    【数値・データの表示】
    - 金額は3桁区切りで表示（例：1,234,567円）
    - パーセンテージは小数点第1位まで表示（例：12.3%）
    - 日付は YYYY年MM月DD日 形式で表示
    
    【回答の構成】
    - まず結論や要点を述べる
    - 必要に応じて詳細データや根拠を示す
    - 追加で確認できることがあれば提案する
    
    【注意事項】
    - 検索結果がない場合は、その旨を明確に伝える
    - 推測や不確実な情報には「〜と考えられます」を使用
    - 個人情報（顧客名、電話番号など）は直接表示しない

tools:
  # Cortex Search（FAQドキュメント）
  - tool_spec:
      type: cortex_search
      name: FAQ_Search
      description: |
        GlacierStyle ECサイトのよくある質問（FAQ）から回答を検索します。
        返品・交換、配送、支払い、会員登録などに関する質問に対応します。

  # Cortex Search（業務マニュアル）
  - tool_spec:
      type: cortex_search
      name: Operation_Manual_Search
      description: |
        カスタマーサポート業務の運営マニュアルから手順や対応方法を検索します。
        クレーム対応、返品処理、エスカレーション手順などの業務フローを参照できます。

  # Cortex Search（音声ログ）
  - tool_spec:
      type: cortex_search
      name: Voice_Log_Search
      description: |
        コールセンターの過去の通話履歴（要約）から類似事例を検索します。
        過去の問い合わせ対応事例やクレーム対応履歴を参照できます。

tool_resources:
  # FAQ検索の設定
  FAQ_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_FAQ
    max_results: 5

  # 業務マニュアル検索の設定
  Operation_Manual_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_OPERATION_MANUALS
    max_results: 5

  # 音声ログ検索の設定
  Voice_Log_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_VOICE_LOGS
    max_results: 5
$$;

In [ ]:
-- ============================================================================
-- 作成したエージェントの確認
-- ============================================================================
SHOW AGENTS IN SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 3. AI Studioでツールを追加（GUI体験）

エージェントが作成されたので、AI Studioで残り2つのツールを追加して完成させます。  
ここでは **Semantic View** と **Cortex Search（SNS）** をGUIで追加します。

---

### 3-1. AI Studioを開く

1. Snowsight → **AI と ML** → **エージェント** をクリック
2. `GLACIER_ANALYTICS_AGENT` をクリック
3. **編集**ボタンをクリック
4. **ツール**タブを開く

> 💡 この時点で、先ほど作成した3つのCortex Searchツール（FAQ、業務マニュアル、音声ログ）が表示されています。

---

### 3-2. Semantic Viewツールの追加

**目的:** 売上・顧客・商品データを自然言語で分析

#### 手順

1. Cortex アナリストの横の「**+ 追加**」をクリック
2. 以下の設定を入力：

| 設定項目 | 値 |
|---------|----|  
| セマンティックビュー | チェックを入れる |
| データベース | `GLACIERSTYLE_DB` |
| スキーマ | `EC_ANALYTICS_SCHEMA` |
| セマンティックビュー | `EC_ANALYSIS_SEMANTIC_VIEW` |
| ツール名 | `EC_Sales_Customer_Analysis` |
| 説明 | GlacierStyle ECサイトの売上、注文、顧客、商品、決済データを分析します。売上推移、カテゴリ別売上、顧客セグメント分析、購買傾向などの質問に回答できます。データは2024年のものです。 |

3. 「**追加**」をクリック

---

### 3-3. Cortex Searchツールの追加（SNS検索）

**目的:** SNS投稿から顧客の声を検索

#### 手順

1. Cortex 検索サービスの横の「**+ 追加**」をクリック
2. 以下の設定を入力：

| 設定項目 | 値 |
|---------|----|  
| データベース | `GLACIERSTYLE_DB` |
| スキーマ | `EC_ANALYTICS_SCHEMA` |
| Cortex Search Service | `SEARCH_SNS_MENTIONS` |
| 最大結果 | `10` |
| ID 列 | `POST_ID` |
| タイトル列 | `POST_ID` |
| ツール名 | `SNS_Mention_Search` |
| Description | SNS（Twitter/Instagram）上のGlacierStyle関連の投稿から顧客の声を検索します。商品の評判、ブランドイメージ、改善要望などのVoC情報を参照できます。 |

3. 「**追加**」をクリック

---

### ✅ エージェント完成！

これで5つのツールがすべて追加されました：

| ツール名 | 種別 | 追加方法 |
|---------|------|----------|
| FAQ_Search | Cortex Search | Notebooks |
| Operation_Manual_Search | Cortex Search | Notebooks |
| Voice_Log_Search | Cortex Search | Notebooks |
| EC_Sales_Customer_Analysis | Semantic View | **AI Studio** |
| SNS_Mention_Search | Cortex Search | **AI Studio** |

---

> **💡 GUIでツールを追加する利点**
> - 設定項目をビジュアルで確認しながら入力できる
> - カラムの選択がドロップダウンで簡単
> - フィルター条件の設定もGUIで可能

## 4. 【オプション】全ツールを含むエージェントの一括作成

> **💡 このセクションはオプションです**  
> 上記のGUI手順でツールを追加した場合は、このセクションをスキップしてください。  
> GUI操作をスキップして一括でエージェントを完成させたい場合にのみ実行してください。

---

以下のSQLを実行すると、5つのツールすべてを含むエージェントが作成されます。

| ツール名 | 種別 | 対象 |
|---------|------|------|
| EC_Sales_Customer_Analysis | Semantic View | 売上・顧客・商品データ |
| FAQ_Search | Cortex Search | FAQドキュメント |
| Operation_Manual_Search | Cortex Search | 業務マニュアル |
| Voice_Log_Search | Cortex Search | 音声ログ要約 |
| SNS_Mention_Search | Cortex Search | SNS投稿 |

In [ ]:
-- ============================================================================
-- 【オプション】Cortex Agent の作成（全ツール含む完全版）
-- ※ GUI操作でツールを追加した場合は実行不要です
-- ※ 一括でエージェントを完成させたい場合のみ実行してください
-- ============================================================================
CREATE OR REPLACE AGENT GLACIER_ANALYTICS_AGENT
  COMMENT = 'GlacierStyle ECサイトの売上・顧客・VoC分析を自然言語で行うエージェントです。'
  PROFILE = '{"display_name": "GLACIER分析エージェント", "color": "blue"}'
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  orchestration: |
    あなたはGlacierStyle ECサイトの分析アシスタントです。
    ユーザーの質問に対して、以下の手順で適切なツールを選択してください。
    
    1. 質問の種類を判断する
       - 売上・注文・顧客・商品に関する数値分析 → EC_Sales_Customer_Analysis（Semantic View）を使用
       - 返品・配送・支払いなどのFAQ → FAQ_Search を使用
       - 業務手順・対応方法 → Operation_Manual_Search を使用
       - 過去の問い合わせ事例 → Voice_Log_Search を使用
       - SNSの評判・口コミ → SNS_Mention_Search を使用
    
    2. 複合的な質問の場合は、複数のツールを順番に使用する
    
    3. 検索結果が不十分な場合は、別のツールを試すか、ユーザーに追加情報を求める
    
    4. データの期間に注意する
       - 売上・注文データは2024年のデータです
       - 「今月」「先月」と言われた場合は、2024年12月・11月として解釈してください
  
  response: |
    以下のルールに従って応答してください。
    
    【口調・スタイル】
    - 丁寧語（です・ます調）で回答する
    - 専門用語は必要に応じて簡単な説明を添える
    - 回答は簡潔にまとめつつ、必要な情報は漏らさない
    
    【数値・データの表示】
    - 金額は3桁区切りで表示（例：1,234,567円）
    - パーセンテージは小数点第1位まで表示（例：12.3%）
    - 日付は YYYY年MM月DD日 形式で表示
    
    【回答の構成】
    - まず結論や要点を述べる
    - 必要に応じて詳細データや根拠を示す
    - 追加で確認できることがあれば提案する
    
    【注意事項】
    - 検索結果がない場合は、その旨を明確に伝える
    - 推測や不確実な情報には「〜と考えられます」を使用
    - 個人情報（顧客名、電話番号など）は直接表示しない

tools:
  # Semantic View（売上・顧客分析）
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: EC_Sales_Customer_Analysis
      description: |
        GlacierStyle ECサイトの売上、注文、顧客、商品、決済データを分析します。
        売上推移、カテゴリ別売上、顧客セグメント分析、購買傾向などの質問に回答できます。
        データは2024年のものです。

  # Cortex Search（FAQドキュメント）
  - tool_spec:
      type: cortex_search
      name: FAQ_Search
      description: |
        GlacierStyle ECサイトのよくある質問（FAQ）から回答を検索します。
        返品・交換、配送、支払い、会員登録などに関する質問に対応します。

  # Cortex Search（業務マニュアル）
  - tool_spec:
      type: cortex_search
      name: Operation_Manual_Search
      description: |
        カスタマーサポート業務の運営マニュアルから手順や対応方法を検索します。
        クレーム対応、返品処理、エスカレーション手順などの業務フローを参照できます。

  # Cortex Search（音声ログ）
  - tool_spec:
      type: cortex_search
      name: Voice_Log_Search
      description: |
        コールセンターの過去の通話履歴（要約）から類似事例を検索します。
        過去の問い合わせ対応事例やクレーム対応履歴を参照できます。

  # Cortex Search（SNS投稿）
  - tool_spec:
      type: cortex_search
      name: SNS_Mention_Search
      description: |
        SNS（Twitter/Instagram）上のGlacierStyle関連の投稿から顧客の声を検索します。
        商品の評判、ブランドイメージ、改善要望などのVoC情報を参照できます。

tool_resources:
  # Semantic Viewの設定
  EC_Sales_Customer_Analysis:
    semantic_view: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.EC_ANALYSIS_SEMANTIC_VIEW

  # FAQ検索の設定
  FAQ_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_FAQ
    max_results: 5

  # 業務マニュアル検索の設定
  Operation_Manual_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_OPERATION_MANUALS
    max_results: 5

  # 音声ログ検索の設定
  Voice_Log_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_VOICE_LOGS
    max_results: 5

  # SNS投稿検索の設定
  SNS_Mention_Search:
    search_service: GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA.SEARCH_SNS_MENTIONS
    max_results: 10
$$;

In [ ]:
-- ============================================================================
-- エージェントの詳細確認（追加されたツール一覧）
-- ============================================================================
DESCRIBE AGENT GLACIER_ANALYTICS_AGENT;

## 5. サンプル質問の設定（オプション）

エージェント画面に表示されるサンプル質問を設定します。  
これはAI Studioの **Settings** タブから設定することもできます。

> **注意**: 本ハンズオンのデータは**2024年のデータ**です。  
> 「今月」「先月」ではなく、具体的な期間（2024年12月など）を指定してください。

---

### AI StudioでのPROFILE設定手順

1. Snowsight → **AI と ML** → **エージェント** をクリック
2. `GLACIER_ANALYTICS_AGENT` をクリック
3. **編集**ボタンをクリック
3. **質問の例** セクションで質問を追加
4. 「**保存**」をクリック

---

### 推奨サンプル質問

- 「2024年12月の売上上位10商品を教えて」
- 「2024年11月と12月の売上を比較して」
- 「返品ポリシーについて教えて」
- 「クレーム対応の手順を教えて」
- 「配送遅延に関する過去の問い合わせ事例を探して」
- 「SNSでネガティブな投稿が多い商品は？」
- 「2024年Q4のカテゴリ別売上構成比を教えて」
- 「新規顧客とリピーターの購買金額の違いは？」
- 「商品の在庫切れに関するFAQを検索して」
- 「SNSでの商品の評判を教えて」

## 6. Snowflake Intelligenceへの公開

作成したエージェントをSnowflake Intelligenceとして公開します。

### 公開手順

1. Snowsight → **AI と ML** → **エージェント** を開く
2. `GLACIER_ANALYTICS_AGENT` をクリック
3. 左上の **+ エージェントを追加** をクリック

## 7. 分析実践

Snowflake Intelligenceを使って、以下のユースケースで分析を実践してみましょう。

---

### 7-1. VoC（Voice of Customer）分析

**目的:** コンタクトセンターログとSNSログから顧客満足度を分析

**試してみる質問:**
- 「ネガティブな問い合わせの内訳を教えて」
- 「配送に関する不満の主な原因は何？」
- 「SNSで最も言及されている商品は？」
- 「クレーム件数が多い曜日や時間帯は？」

---

### 7-2. 売上分析

**目的:** 売上データとアクセスログの統合分析

**試してみる質問:**
- 「2024年12月の売上が前月比で10%以上減少したカテゴリは？」
- 「新規顧客とリピーターの購買金額の違いは？」
- 「週末と平日で売れ筋商品に違いはある？」
- 「カテゴリ別の売上構成比を教えて」

---

### 7-3. 複合分析（構造化 + 非構造化）

**目的:** 売上データとVoCを組み合わせた分析

**試してみる質問:**
- 「売上が多い商品について、SNSでの評判を教えて」
- 「返品が多い商品カテゴリと、その理由を教えて」
- 「クレームが多い商品の特徴を分析して」

## 8. モニタリング

エージェントの動作をモニタリングして、精度向上に役立てます。

---

### モニタリング手順

1. Snowsight → **AI と ML** → **エージェント** を開く
2. `GLACIER_ANALYTICS_AGENT` をクリック
3. **モニタリング** タブを開く

---

### 確認できる情報

- **タイムスタンプ**: いつエージェントが利用されたか
- **ユーザー入力**: 実際の質問内容
- **プランニング内容**: AIがどのツールを選択したか
- **ツール呼び出し**: 実際に使用されたツールとパラメータ
- **ツール応答**: 各ツールからの返却データ
- **最終応答**: ユーザーに表示された回答

---

### 改善アクション例

| 観察された問題 | 改善アクション |
|---------------|---------------|
| 間違ったツールが選択される | オーケストレーション手順を見直し |
| SQL変換が不正確 | Semantic Viewの定義を改善、確認済みクエリを追加 |
| 検索結果が不十分 | Cortex Searchのmax_resultsを増やす |
| 応答が冗長 | 応答手順を調整 |

## まとめ

このノートブックでは、Snowflake Cortex Agentを作成し、Snowflake Intelligenceとして公開しました。

### 作成したエージェント

| 項目 | 値 |
|-----|----|  
| エージェント名 | GLACIER_ANALYTICS_AGENT |
| 表示名 | GLACIER分析エージェント |
| オーケストレーションモデル | auto（最新モデル自動選択） |

### 追加したツール

| ツール名 | 種別 | 対象 | 追加方法 |
|---------|------|------|----------|
| FAQ_Search | Cortex Search | FAQドキュメント | Notebooks |
| Operation_Manual_Search | Cortex Search | 業務マニュアル | Notebooks |
| Voice_Log_Search | Cortex Search | 音声ログ要約 | Notebooks |
| EC_Sales_Customer_Analysis | Semantic View | 売上・顧客・商品データ | AI Studio |
| SNS_Mention_Search | Cortex Search | SNS投稿 | AI Studio |

### Cortex Agentのポイント

- **ツールの追加方法**: `CREATE OR REPLACE AGENT` で一括定義、またはAI StudioのGUIから追加
- **ALTER AGENTコマンドは存在しない**: ツール追加後は再作成が必要（GUIやREST APIでも内部で再作成）
- **Snowflake Intelligence**: 公開することでエンドユーザーが簡単にアクセス可能
- **ハイブリッドアプローチ**: Notebooksで基本を作成 → GUIで仕上げが効率的

### 次のステップ

- **Part 6**: Streamlit in Snowflakeによるカスタムアプリ開発
  - 広告クリエイティブ分析ダッシュボード
  - VoC分析ダッシュボード
  - マルチモーダル検索UI